In [1]:
# Now to download population projections to 2071 data:)

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl


In [2]:
Current_Path = Path.cwd()

Path_Root = Current_Path.parent.parent

Data_Path = Path_Root/"data"/"Forecasting_Data"
File_Path = Data_Path/"MediumSeries_ASPData_2171.xlsx"

Data_Path.mkdir(parents=True, exist_ok=True)
Url = (
    "https://www.abs.gov.au/statistics/people/population/"
    "population-projections-australia/2022-base-2071/"
    "3222_Table_B2.xlsx"
)

Response = requests.get(
    Url,
    timeout=60,
)

Response.raise_for_status()

with open(File_Path, "wb") as File:
    File.write(Response.content)


print(f"Downloaded successfully to: {File_Path}")


Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/Forecasting_Data/MediumSeries_ASPData_2171.xlsx


In [3]:
###############################################################
# Now to read the excel sheets 'data1' and 'data2' into as
# pandas data frames, then combine the data into 1
################################################################

Data1 = pd.read_excel(
    File_Path,
    sheet_name="Data1",
    header=None,
)

Data2 = pd.read_excel(
    File_Path,
    sheet_name="Data2",
    header=None,
)

In [4]:
# Now for the combining :)

Data2 = Data2.iloc[:, 2:]

Combined_Data = pd.concat(
    [Data1, Data2],
    axis=1,
)


In [5]:
Csv1_Data_Path = Path_Root/"data"/"Forecasting_Data"
Csv1_File_Path = Csv1_Data_Path/"MediumSeries_ASPData_2171_Uncleaned.csv"

Csv1_Data_Path.mkdir(parents=True, exist_ok=True)

Combined_Data.to_csv(
    Csv1_File_Path,
    index=False,
    header=False,
)

print(f"Saved combined data to: {Csv1_File_Path}")
print(f"Shape: {Combined_Data.shape}")

Saved combined data to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/Forecasting_Data/MediumSeries_ASPData_2171_Uncleaned.csv
Shape: (60, 258)


In [6]:
Uncleaned_Data = pd.read_csv("../../data/Forecasting_Data"
                             "/MediumSeries_ASPData_2171_Uncleaned.csv"
                             )


In [7]:
Series_Headers = Uncleaned_Data.columns[1:]

Clean_Headers = ["Year"]

for Header in Series_Headers:

    Parts = [
        Part.strip()
        for Part in str(Header).split(";")
        if Part.strip()
    ]

    Sex = Parts[3]

    Age = Parts[4].replace(
        " ",
        "_",
    )

    Column_Name = f"{Sex}_Age_{Age}"

    Clean_Headers.append(Column_Name)

In [8]:
# Now apply coloum name changes :)

Uncleaned_Data.columns = Clean_Headers

In [9]:
# Now to remove the meta data 

Cleaned_Data = Uncleaned_Data.iloc[9:].copy()



In [10]:
# Convert values such as:
# 2022-06-30 -> 2022
Cleaned_Data["Year"] = (
    pd.to_datetime(Cleaned_Data["Year"])
    .dt.year
)


In [14]:
Population_Columns = Cleaned_Data.columns.drop(
    "Year"
)

Cleaned_Data[Population_Columns] = (
    Cleaned_Data[Population_Columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)


In [ ]:
Cleaned_Data

In [16]:
Cleaned_Data = Cleaned_Data.reset_index(
    drop=True
)

In [ ]:
Csv2_File_Path = Csv1_Data_Path/"MediumSeries_ASPData_2171_Cleaned.csv"
Cleaned_Data.to_csv(
    Csv2_File_Path,
    index=False,
)
print(
    f"Cleaned data saved to: {Csv2_File_Path}"
)

print(
    f"Rows: {len(Cleaned_Data)}"
)

print(
    f"Columns: {len(Cleaned_Data.columns)}"
)

print(
    f"Years: "
    f"{Cleaned_Data['Year'].min()}-"
    f"{Cleaned_Data['Year'].max()}"
)

print(Cleaned_Data.head())